In [ ]:
%load_ext watermark


In [ ]:
import os

from IPython.display import display
from teeplot import teeplot as tp

import pylib  # noqa: F401
from pyfonts import load_google_font


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-02-04-complexity-drivers")
teeplot_subdir


## Example Plot


In [ ]:
# curved text adapted from https://stackoverflow.com/a/44521963


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from matplotlib.path import Path
from matplotlib import text as mtext
import math

# --- 1. The CurvedText Class ---

class CurvedText(mtext.Text):
    """
    A text object that follows an arbitrary curve.
    """
    def __init__(self, x, y, text, axes, **kwargs):
        super(CurvedText, self).__init__(x[0],y[0],' ', **kwargs)

        axes.add_artist(self)

        self.__x = x
        self.__y = y
        self.__zorder = self.get_zorder()

        self.__Characters = []
        for c in text:
            if c == ' ':
                t = mtext.Text(0,0,'a')
                t.set_alpha(0.0)
            else:
                t = mtext.Text(0,0,c, **kwargs)

            t.set_ha('center')
            t.set_rotation(0)
            t.set_zorder(self.__zorder +1)

            self.__Characters.append((c,t))
            axes.add_artist(t)

    def set_zorder(self, zorder):
        super(CurvedText, self).set_zorder(zorder)
        self.__zorder = self.get_zorder()
        for c,t in self.__Characters:
            t.set_zorder(self.__zorder+1)

    def draw(self, renderer, *args, **kwargs):
        self.update_positions(renderer)

    def update_positions(self, renderer):
        xlim = self.axes.get_xlim()
        ylim = self.axes.get_ylim()
        figW, figH = self.axes.get_figure().get_size_inches()
        bbox = self.axes.get_position()
        w, h = bbox.width, bbox.height

        dx = xlim[1]-xlim[0] if xlim[1]!=xlim[0] else 1.0
        dy = ylim[1]-ylim[0] if ylim[1]!=ylim[0] else 1.0
        aspect = ((figW * w)/(figH * h))*(dy/dx)

        trans = self.axes.transData.transform
        pts = np.column_stack([self.__x, self.__y])
        trans_pts = trans(pts)
        x_fig, y_fig = trans_pts[:,0], trans_pts[:,1]

        x_fig_dist = (x_fig[1:]-x_fig[:-1])
        y_fig_dist = (y_fig[1:]-y_fig[:-1])
        r_fig_dist = np.sqrt(x_fig_dist**2+y_fig_dist**2)
        l_fig = np.insert(np.cumsum(r_fig_dist),0,0)

        rads = np.arctan2((y_fig[1:] - y_fig[:-1]),(x_fig[1:] - x_fig[:-1]))
        degs = np.rad2deg(rads)

        rel_pos = 0

        for c,t in self.__Characters:
            t.set_rotation(0)
            t.set_va('center')
            bbox1  = t.get_window_extent(renderer=renderer)
            w_char = bbox1.width
            h_char = bbox1.height

            if rel_pos + w_char/2 > l_fig[-1]:
                t.set_alpha(0.0)
                rel_pos += w_char
                continue
            elif c != ' ':
                t.set_alpha(1.0)

            target_len = rel_pos + w_char/2
            candidates = np.where(target_len <= l_fig)[0]

            if len(candidates) == 0:
                ir = len(l_fig)-1
                il = ir - 1
            else:
                ir = candidates[0]
                il = ir - 1

            if il < 0:
                il = 0
                ir = 1

            used = l_fig[il] - rel_pos
            dist_from_il = (rel_pos + w_char/2) - l_fig[il]

            seg_len = r_fig_dist[il] if il < len(r_fig_dist) else 1.0
            if seg_len == 0: seg_len = 1.0

            fraction = dist_from_il / seg_len

            x_new = self.__x[il]+fraction*(self.__x[ir]-self.__x[il])
            y_new = self.__y[il]+fraction*(self.__y[ir]-self.__y[il])

            t.set_va('center')

            t.set_position(np.array([x_new,y_new]))
            t.set_rotation(degs[il])

            rel_pos += w_char

# --- 2. Plotting Logic ---

try:
    font = load_google_font('Merriweather', weight='regular')
except NameError:
    from matplotlib.font_manager import FontProperties
    font = FontProperties(family='serif')

def get_text_arc_params(text, center_angle, radius, font_size, spacing=1.3):
    """
    Calculates start and end angles for a text string without drawing it.
    Used to determine the width of the underline arc.
    """
    points_per_data_unit = (8 * 72) / 2.0
    char_width_data = (0.5 * font_size) / points_per_data_unit
    effective_char_width = char_width_data * spacing
    arc_length = effective_char_width * len(text)
    total_angle_rad = arc_length / radius

    start_angle = center_angle + total_angle_rad / 2
    end_angle = center_angle - total_angle_rad / 2
    return start_angle, end_angle

def draw_curved_text_adapter(ax, text, center_angle, radius, font_size, color='black',
                             spacing=1.3):
    """
    Adapts the parameters to use the CurvedText class.
    """
    points_per_data_unit = (8 * 72) / 2.0
    char_width_data = (0.5 * font_size) / points_per_data_unit
    effective_char_width = char_width_data * spacing
    arc_length = effective_char_width * len(text)
    total_angle_rad = arc_length / radius

    start_angle = center_angle + total_angle_rad / 2
    end_angle = center_angle - total_angle_rad / 2

    N = 100
    theta = np.linspace(start_angle, end_angle, N)
    x_curve = radius * np.cos(theta)
    y_curve = radius * np.sin(theta)

    CurvedText(
        x=x_curve,
        y=y_curve,
        text=text,
        axes=ax,
        fontsize=font_size,
        fontproperties=font,
        color=color,
        va='center',
        ha='center'
    )

def draw_bezier_connection(ax, start, end, color='black', linewidth=1, linestyle='-', tension=0.4):
    p0 = start
    p3 = end
    p1 = (start[0] * tension, start[1] * tension)
    p2 = (end[0] * tension, end[1] * tension)

    verts = [p0, p1, p2, p3]
    codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]

    path = Path(verts, codes)
    patch = patches.PathPatch(path, facecolor='none', edgecolor=color, lw=linewidth, ls=linestyle, zorder=1)
    ax.add_patch(patch)

def draw_plot():
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_aspect('equal')
    ax.axis('off')

    limit = 1.15
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit - 0.1, limit + 0.25)

    r = 0.6
    # Moved outer row further out to prevent overlap (approx 10% shift from 0.88 -> 0.95)
    label_r_inner = 0.78
    label_r_outer = 0.92
    delta = 12

    c_pheno = '#5CAEAE'
    c_geno = "#7C6AD6"
    c_crypt = "#7A6AD6"

    ang_p_center = np.deg2rad(145)
    ang_g_center = np.deg2rad(90)
    ang_c_center = np.deg2rad(35)

    ang_p_label = np.deg2rad(162)
    ang_g_label = np.deg2rad(95.5)
    ang_c_label = np.deg2rad(24)

    ang_nc = np.deg2rad(217.5)
    ang_fitness = np.deg2rad(270)
    ang_drift = np.deg2rad(322.5)

    nodes = {
        'p_early': (r * np.cos(ang_p_center + np.deg2rad(delta)), r * np.sin(ang_p_center + np.deg2rad(delta))),
        'p_all': (r * np.cos(ang_p_center - np.deg2rad(delta)), r * np.sin(ang_p_center - np.deg2rad(delta))),
        'g_early': (r * np.cos(ang_g_center + np.deg2rad(delta)), r * np.sin(ang_g_center + np.deg2rad(delta))),
        'g_all': (r * np.cos(ang_g_center - np.deg2rad(delta)), r * np.sin(ang_g_center - np.deg2rad(delta))),
        'c_early': (r * np.cos(ang_c_center + np.deg2rad(delta)), r * np.sin(ang_c_center + np.deg2rad(delta))),
        'c_all': (r * np.cos(ang_c_center - np.deg2rad(delta)), r * np.sin(ang_c_center - np.deg2rad(delta))),
        'nc': (r * np.cos(ang_nc), r * np.sin(ang_nc)),
        'fitness': (r * np.cos(ang_fitness), r * np.sin(ang_fitness)),
        'drift': (r * np.cos(ang_drift), r * np.sin(ang_drift))
    }

    marker_size = 625
    square_size = marker_size * 0.75
    font_size_top = 34
    font_size_bottom = 31

    bottom_items = [
        ('nc', 'niche\nconst'),
        ('fitness', 'fitness'),
        ('drift', 'drift\n(time)')
    ]

    for name, label in bottom_items:
        x, y = nodes[name]
        ax.scatter(x, y, s=square_size, marker='D', color='black', edgecolors='black', zorder=10, linewidth=2)
        offset_y = -0.07
        if '\n' in label:
             offset_y = -0.09
        ax.text(x, y + offset_y, label, ha='center', va='top', fontsize=font_size_bottom, fontproperties=font, color='black')

    pairs = [
        ('p_early', 'p_all', 'Phenotype ', 'Complexity', ang_p_label, c_pheno, 0.0),
        ('g_early', 'g_all', 'Genotype ', 'Complexity', ang_g_label, c_geno, 0.04),
        ('c_early', 'c_all', '+Near-neutral', 'G. Complexity', ang_c_label, c_crypt, 0.04)
    ]

    for early, all_n, label_top, label_bot, ang, color, span_red in pairs:
        # Draw Nodes
        ax.scatter(*nodes[early], s=marker_size, marker='o', facecolor='white', edgecolor=color, zorder=10, linewidth=2.5)
        ax.scatter(*nodes[all_n], s=marker_size, marker='o', facecolor=color, edgecolor=color, zorder=10, linewidth=2.5)

        # Draw Text
        draw_curved_text_adapter(ax, label_top, ang, label_r_outer, font_size_top, color=color, spacing=1.2)
        draw_curved_text_adapter(ax, label_bot, ang, label_r_inner, font_size_top, color=color, spacing=1.2)

        # --- Thick Underline / Background Arc ---
        # Position: Middle of the two rows
        mid_r = (label_r_inner + label_r_outer) / 2

        # Calculate angular width needed based on the TOP label (usually wider)
        s_ang, e_ang = get_text_arc_params(label_top, ang, mid_r, font_size_top, spacing=1.2)

        # Apply span reduction from params
        arc_start_rad = s_ang - span_red
        arc_end_rad = e_ang + span_red

        # Convert to degrees for patches.Arc
        theta1 = np.rad2deg(arc_end_rad)
        theta2 = np.rad2deg(arc_start_rad)

        # Calculate thickness to cover the gap + padding
        # distance between rows is (outer - inner)
        dist_between = label_r_outer - label_r_inner
        # Scale factor manually tuned to look like a solid block behind both
        arc_lw = dist_between * 600  # Multiplier to convert data units to points approximately

        arc = patches.Arc((0,0), mid_r*2, mid_r*2, angle=0,
                          theta1=theta1, theta2=theta2, color=color,
                          lw=arc_lw, alpha=0.15, zorder=5, capstyle='round')
        ax.add_patch(arc)

    nc_pheno_start = (nodes['nc'][0], nodes['nc'][1] + 0.025)
    nc_geno_start = (nodes['nc'][0], nodes['nc'][1] - 0.025)

    thick_lw = 10.4
    mid_lw = 5.2

    draw_bezier_connection(ax, nc_pheno_start, nodes['p_early'], color=c_pheno, linewidth=1.5, linestyle='--', tension=0.4)
    draw_bezier_connection(ax, nc_pheno_start, nodes['p_all'], color=c_pheno, linewidth=thick_lw, linestyle='-', tension=0.4)

    draw_bezier_connection(ax, nc_geno_start, nodes['g_early'], color=c_geno, linewidth=thick_lw, linestyle='-', tension=0.4)
    draw_bezier_connection(ax, nc_geno_start, nodes['g_all'], color=c_geno, linewidth=1.5, linestyle='--', tension=0.4)

    draw_bezier_connection(ax, nodes['drift'], nodes['c_early'], color=c_crypt, linewidth=1.5, linestyle='--', tension=0.4)
    draw_bezier_connection(ax, nodes['drift'], nodes['c_all'], color=c_crypt, linewidth=thick_lw, linestyle='-', tension=0.4)

    # --- Legend ---
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='w', markeredgecolor='black', markeredgewidth=2.6, markersize=20, label='early'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markeredgecolor='black', markeredgewidth=2.6, markersize=20, label='all'),
        Line2D([0], [0], color='black', lw=2.6, linestyle='--', label='p < 0.05'),
        Line2D([0], [0], color='black', lw=2.6, linestyle='-', label='p < 0.01'),
        Line2D([0], [0], color='black', lw=mid_lw, label='$R^2 = 0.2$'),
        Line2D([0], [0], color='black', lw=thick_lw, label='$R^2 = 0.4$')
    ]

    font2 = font.copy()
    font2.set_size(23)
    ax.legend(handles=legend_elements, loc='upper center',
                    bbox_to_anchor=(0.5, 1.05),
                    ncol=3, frameon=False,
                    columnspacing=1.5,
                    handletextpad=0.5,
                    labelspacing=1.5,
                    prop=font2)

    plt.tight_layout()

# draw_plot()


In [ ]:
tp.tee(
    draw_plot,
    teeplot_subdir=teeplot_subdir,
)
